In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
import os
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [ ]:
train_dir = os.path.join(path, "PlantVillage", "train")
test_dir  = os.path.join(path, "PlantVillage", "test")

print(os.listdir(train_dir)[:3])
print(os.listdir(test_dir)[:3])

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
])

In [ ]:
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
test_dataset  = datasets.ImageFolder(test_dir,  transform=test_transform)

class_names = train_dataset.classes
print("classes:", class_names)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=0)

In [ ]:
imgs, labels = next(iter(train_loader))

plt.figure(figsize=(10,4))
for i in range(8):
    plt.subplot(2,4,i+1)
    plt.imshow(imgs[i].permute(1,2,0))
    plt.title(class_names[labels[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Write your code here
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

class CNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.pool = nn.MaxPool2d(2,2)

        self.c1 = nn.Conv2d(3,  32, 3, padding=1); self.b1 = nn.BatchNorm2d(32)
        self.c2 = nn.Conv2d(32, 64, 3, padding=1); self.b2 = nn.BatchNorm2d(64)
        self.c3 = nn.Conv2d(64, 64, 3, padding=1); self.b3 = nn.BatchNorm2d(64)
        self.c4 = nn.Conv2d(64, 128,3, padding=1); self.b4 = nn.BatchNorm2d(128)
        self.c5 = nn.Conv2d(128,256,3, padding=1); self.b5 = nn.BatchNorm2d(256)

        # 32 -> 16 -> 8 -> 4 -> 2 -> 1
        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.b1(self.c1(x))))
        x = self.pool(F.relu(self.b2(self.c2(x))))
        x = self.pool(F.relu(self.b3(self.c3(x))))
        x = self.pool(F.relu(self.b4(self.c4(x))))
        x = self.pool(F.relu(self.b5(self.c5(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = CNN(num_classes=3).to(device)
print(model)

In [ ]:
# Write your code here
def train_one_epoch(model, loader, loss_fn, opt, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        opt.zero_grad()
        out = model(imgs)
        loss = loss_fn(out, labels)
        loss.backward()
        opt.step()

        total_loss += loss.item() * imgs.size(0)
        pred = out.argmax(1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

In [ ]:
def validate_one_epoch(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            loss = loss_fn(out, labels)

            total_loss += loss.item() * imgs.size(0)
            pred = out.argmax(1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

In [ ]:
# Write your code here
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 5
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(epochs):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, opt, device)
    va_loss, va_acc = validate_one_epoch(model, test_loader, loss_fn, device)

    train_losses.append(tr_loss); val_losses.append(va_loss)
    train_accs.append(tr_acc);   val_accs.append(va_acc)

    print(f"Epoch {epoch+1}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {va_loss:.4f} acc {va_acc:.4f}")

In [ ]:
plt.figure()
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("epoch"); plt.ylabel("loss")
plt.legend(); plt.show()

plt.figure()
plt.plot(train_accs, label="train")
plt.plot(val_accs, label="val")
plt.xlabel("epoch"); plt.ylabel("acc")
plt.legend(); plt.show()

In [ ]:
# Write your code here
class CNNRes(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.pool = nn.MaxPool2d(2,2)

        self.c1 = nn.Conv2d(3,  32, 3, padding=1); self.b1 = nn.BatchNorm2d(32)
        self.c2 = nn.Conv2d(32, 64, 3, padding=1); self.b2 = nn.BatchNorm2d(64)
        self.c3 = nn.Conv2d(64, 64, 3, padding=1); self.b3 = nn.BatchNorm2d(64)
        self.c4 = nn.Conv2d(64, 128,3, padding=1); self.b4 = nn.BatchNorm2d(128)
        self.c5 = nn.Conv2d(128,256,3, padding=1); self.b5 = nn.BatchNorm2d(256)

        self.skip = nn.Conv2d(64, 64, 1)  # align channels for sum

        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.b1(self.c1(x))))      # 32->16
        x2 = self.pool(F.relu(self.b2(self.c2(x))))     # 16->8 (save)

        x3 = self.pool(F.relu(self.b3(self.c3(x2))))    # 8->4

        # make skip match x3 spatial: x2(8)->pool->4
        s = self.pool(self.skip(x2))                    # 8->4

        x = x3 + s                                      # sum skip

        x = self.pool(F.relu(self.b4(self.c4(x))))      # 4->2
        x = self.pool(F.relu(self.b5(self.c5(x))))      # 2->1

        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = CNNRes(num_classes=3).to(device)
print(model)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 5
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(epochs):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, opt, device)
    va_loss, va_acc = validate_one_epoch(model, test_loader, loss_fn, device)

    train_losses.append(tr_loss); val_losses.append(va_loss)
    train_accs.append(tr_acc);   val_accs.append(va_acc)

    print(f"Epoch {epoch+1}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {va_loss:.4f} acc {va_acc:.4f}")

In [ ]:
plt.figure()
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("epoch"); plt.ylabel("loss")
plt.legend(); plt.show()

plt.figure()
plt.plot(train_accs, label="train")
plt.plot(val_accs, label="val")
plt.xlabel("epoch"); plt.ylabel("acc")
plt.legend(); plt.show()